## Project__1: Single Agent Using LangChain

- simple answer question and get the lasted weather condition

### Step 1: Import necessary Libraries

In [42]:
# =====================
# Standard Library
# ====================
import os

# ======================
# Third-Party Libraries
# ======================
import requests
import certifi

# ============================
# Environment Variables
# ============================
from dotenv import load_dotenv

# =============================
# LangChain
# ==============================
from langchain_groq import ChatGroq
from langchain.tools import tool

# ============================
# Tavily Search
# =============================
from langchain_tavily import TavilySearch

In [43]:
# import the agent relative library
from langchain.agents import create_agent

In [44]:
# =============================
# Load Env Variables
# ==============================
# os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


### Step 2: Initial the Tavily Tools

In [45]:
search_tool = TavilySearch(max_results=3)  

In [46]:
# test the search_tools
result = search_tool.invoke("What is the Capital of Bangladesh?")

print(result)

{'query': 'What is the Capital of Bangladesh?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.worldatlas.com/articles/what-is-the-capital-of-bangladesh.html', 'title': 'What Is The Capital Of Bangladesh?', 'content': 'Bangladesh is a country in South Asia bordering India and Burma covering an area of 148,460 km2 and is slightly smaller than the state of Iowa. The population estimate for 2016 is 156,186,882. Bangladesh in the local language means the country of Bengal with most of the population considering themselves as Bengali. Lying along the east bank of the Buriganga River in the heart of the Bengal delta is Dhaka, the capital of Bangladesh. Dhaka is the largest city in Bangladesh and is the cultural and [...] Ever since the city was made the capital of Bangladesh, Dhaka has grown in population, area coverage, and social and economic diversity. Dhaka presently is the most populous and industrialized area in the country. There are differ

### Create the Custom Tools

- get the weather information

In [47]:
from langchain.tools import tool
import requests

In [48]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    # convert to json
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

### Step 3: Initial the LLM

In [49]:

# Initial the LLM
# ==============================
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1
)

In [27]:
# test the llm
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI go to therapy?\n\nBecause it had too many unresolved *neural* issues!', additional_kwargs={'reasoning_content': 'We need to produce a joke about AI. The user just says "Tell me a joke about AI". We can produce a short joke. Should be appropriate. Let\'s produce a witty one.'}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 77, 'total_tokens': 145, 'completion_time': 0.121331112, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003710878, 'prompt_tokens_details': None, 'queue_time': 0.211634609, 'total_time': 0.12504199}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a111ac-48c3-7aa0-b58f-42ee5d8044fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 68, 'total_tokens': 145, 'output_token_details': {'reasoning': 39}})

### Step 4: Define the Prompt

In [ ]:
# ==========================================
# PROMPT
# ==========================================


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} template='\nAnswer the following question as best you can.\n\nYou have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: {input}\nThought: I need to determine what to do.\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\nThought: I now know the final answer.\nFinal Answer: the final answer to the original question\n\nBegin!\n\nQuestion: {input}\nThought: {agent_scratchpad}\n'


### Step 5: Define the list of prompt

In [50]:
tools = [search_tool, get_weather_data]

### Step 6: Creating the Agent

In [51]:

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful and reliable AI assistant.

Your responsibilities:
1. Answer general questions directly when you already have sufficient knowledge.
2. Use the Tavily search tool when the user requests current, recent, or web-based information.
3. Use the weather tool when the user asks about current weather conditions for a city.
4. Do not use a tool when it is unnecessary.
5. Provide a clear, concise, and accurate final answer based on the available information.
"""
)

In [ ]:
# # 2. Create executor
# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True
# )



In [52]:
# ==========================================
# RUN AGENT
# ==========================================

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of Bangladesh and what is weather of right now"
        }
    ]
})

In [53]:
# ==========================================
# DISPLAY FINAL ANSWER
# ==========================================

print(result["messages"][-1].content)

The capital of Bangladesh is **Dhaka**.  
Current weather in Dhaka:  
- **Temperature:** 30 °C  
- **Condition:** Smoky haze  
- **Humidity:** 60 %


In [36]:
# ==========================================
# RUN AGENT
# ==========================================

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Tell me the lasted news about the Iran and USA War"
        }
    ]
})


In [37]:
# ==========================================
# DISPLAY FINAL ANSWER
# ==========================================

print(result["messages"][-1].content)

**Latest developments in the Iran‑U.S. conflict (as of 5 Oct 2026)**  

| Date | Source | Key points |
|------|--------|------------|
| **5 Oct 2026** | *The Independent* | • U.S. forces are reportedly “fleeing” 1,000 km from Iran’s borders, according to a commander’s claim. <br>• President Donald Trump has publicly added “Iran is doing drugs” to the list of reasons for the war, alongside nuclear concerns and a call for regime change. <br>• Iran has summoned the Norwegian ambassador over alleged “unauthorised” Starlink operations, accusing the U.S. and Israel of using the satellite network to advance interventionist agendas. |
| **21 Sep 2026** | *Al Jazeera* (live‑blog) | • Tehran warns the U.S. that it will retaliate “without limitations” if Washington plans new strikes. <br>• The U.S. President says the only options are to “obliterate Iran” or reach a deal. |
| **20 Sep 2026** | *Al Jazeera* | • Iran’s security chief Mohsen Rezaei outlined new conditions for ending the war, echoing 